In [ ]:
# ============================================================
# Cell 1. 設定本機環境與路徑
# ============================================================

# 第一次在本機執行前，請先在終端機安裝依賴：
#   pip install -r requirements.txt

import os
import json
import random
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# 1.1 固定 Random Seed
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 1.2 設定裝置
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置：", device)

if torch.cuda.is_available():
    print("GPU 名稱：", torch.cuda.get_device_name(0))
else:
    print("目前沒有偵測到 GPU，會使用 CPU。訓練大型 RoBERTa 會比較慢。")

# ============================================================
# 1.3 設定本機資料夾與檔案路徑
# ============================================================

BASE_DIR = Path.cwd().resolve()
DATA_DIR = BASE_DIR / "data"
PROJECT_DIR = BASE_DIR / "outputs"
MODEL_DIR = BASE_DIR / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "vpesg_4k_train_1000.json"
LEGACY_RAW_DATA_PATH = DATA_DIR / "vpesg4k_train_1000.json"

DATA_URL = "https://raw.githubusercontent.com/veripromiseesg/veripromiseesgdataset/ac91c1c8b5d116edf6fc44cccc1ee3b618f5a207/vpesg4ktrain1000v1.json"

SAVE_MODEL_PATH = MODEL_DIR / "best_single_roberta_task13.pt"
SAVE_SPLIT_PATH = PROJECT_DIR / "task13_train_valid_test_split.json"
SAVE_HISTORY_PATH = PROJECT_DIR / "task13_training_history.json"

# ============================================================
# 1.4 如果資料不存在，就下載官方訓練資料
# ============================================================

if not RAW_DATA_PATH.exists() and LEGACY_RAW_DATA_PATH.exists():
    RAW_DATA_PATH = LEGACY_RAW_DATA_PATH

if not RAW_DATA_PATH.exists():
    print("找不到資料，開始下載到：", RAW_DATA_PATH)
    urlretrieve(DATA_URL, RAW_DATA_PATH)
    print("資料下載完成")
else:
    print("資料已存在，不需要重新下載")

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"找不到資料檔：{RAW_DATA_PATH}")

# 後續 cell 多數使用 os.path / json / torch；轉成字串可避免跨套件 Path 相容性問題。
BASE_DIR = str(BASE_DIR)
DATA_DIR = str(DATA_DIR)
PROJECT_DIR = str(PROJECT_DIR)
MODEL_DIR = str(MODEL_DIR)
RAW_DATA_PATH = str(RAW_DATA_PATH)
SAVE_MODEL_PATH = str(SAVE_MODEL_PATH)
SAVE_SPLIT_PATH = str(SAVE_SPLIT_PATH)
SAVE_HISTORY_PATH = str(SAVE_HISTORY_PATH)

print("\n路徑設定完成：")
print("BASE_DIR          =", BASE_DIR)
print("PROJECT_DIR       =", PROJECT_DIR)
print("MODEL_DIR         =", MODEL_DIR)
print("RAW_DATA_PATH     =", RAW_DATA_PATH)
print("SAVE_MODEL_PATH   =", SAVE_MODEL_PATH)
print("SAVE_SPLIT_PATH   =", SAVE_SPLIT_PATH)
print("SAVE_HISTORY_PATH =", SAVE_HISTORY_PATH)


In [ ]:
# ============================================================
# Cell 2. 讀取原始資料與檢查欄位
# ============================================================

# 讀取官方 1000 筆訓練資料
with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("原始資料型態：", type(raw_data))
print("原始資料筆數：", len(raw_data))

# 檢查第一筆資料
print("\n第一筆資料 keys：")
print(list(raw_data[0].keys()))

print("\n第一筆資料完整內容：")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2)[:2000])

# ============================================================
# 2.1 檢查必要欄位是否存在
# ============================================================

REQUIRED_COLUMNS = [
    "id",
    "data",
    "promise_status",
    "promise_string",
    "verification_timeline",
    "evidence_status",
    "evidence_string",
    "evidence_quality"
]

missing_cols = []

for col in REQUIRED_COLUMNS:
    if col not in raw_data[0]:
        missing_cols.append(col)

if len(missing_cols) == 0:
    print("\n✅ 必要欄位都存在")
else:
    print("\n⚠️ 缺少欄位：", missing_cols)

# ============================================================
# 2.2 快速查看幾筆文字與標籤
# ============================================================

print("\n前 3 筆資料摘要：")

for i in range(3):
    item = raw_data[i]
    print("\n" + "=" * 80)
    print("index:", i)
    print("id:", item.get("id"))
    print("data:", str(item.get("data", ""))[:300])
    print("promise_status:", item.get("promise_status"))
    print("promise_string:", item.get("promise_string"))
    print("verification_timeline:", item.get("verification_timeline"))
    print("evidence_status:", item.get("evidence_status"))
    print("evidence_string:", item.get("evidence_string"))
    print("evidence_quality:", item.get("evidence_quality"))

In [ ]:
# ============================================================
# Cell 3. 定義官方權重、任務欄位、標籤與文字輸入格式
# ============================================================

# ============================================================
# 3.1 官方評分權重
# ============================================================

OFFICIAL_WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15
}

print("官方評分權重：")
for field, weight in OFFICIAL_WEIGHTS.items():
    print(f"{field}: {weight}")

# ============================================================
# 3.2 這次 RoBERTa 只訓練任務 1 與任務 3
# ============================================================

TASK_FIELDS = [
    "promise_status",
    "evidence_status"
]

LABEL_LISTS = {
    "promise_status": ["Yes", "No"],
    "evidence_status": ["Yes", "No", "N/A"]
}

label2id = {
    field: {label: i for i, label in enumerate(labels)}
    for field, labels in LABEL_LISTS.items()
}

id2label = {
    field: {i: label for i, label in enumerate(labels)}
    for field, labels in LABEL_LISTS.items()
}

num_labels_dict = {
    field: len(labels)
    for field, labels in LABEL_LISTS.items()
}

print("\nRoBERTa 訓練任務欄位：", TASK_FIELDS)

print("\nlabel2id：")
print(label2id)

print("\nid2label：")
print(id2label)

print("\nnum_labels_dict：")
print(num_labels_dict)

# ============================================================
# 3.3 定義文字輸入格式：只使用 data
# ============================================================

def clean_text(value):
    """
    將 None / 空字串 / nan 字串清掉。
    """
    if value is None:
        return ""

    value = str(value).strip()

    if value == "":
        return ""

    if value.lower() == "nan":
        return ""

    return value


def get_text(example):
    """
    給 RoBERTa 的輸入文字。

    照之前做法：
    只使用官方原始文本 data，
    不使用 promise_string / evidence_string，
    避免欄位提示造成分數虛高。
    """
    data_text = clean_text(example.get("data", ""))

    if data_text:
        return data_text

    raise KeyError(f"找不到 data 欄位，目前 keys = {example.keys()}")

# ============================================================
# 3.4 測試 get_text()
# ============================================================

print("\n文字輸入範例：")
print(get_text(raw_data[0])[:1000])

print("\n文字長度範例：")
for i in range(3):
    text = get_text(raw_data[i])
    print(f"index {i}: {len(text)} characters")

In [ ]:
# ============================================================
# Cell 4. 檢查各任務標籤分布，特別確認 Misleading 筆數
# ============================================================

from collections import Counter

# ============================================================
# 4.1 檢查 RoBERTa 要訓練的兩個任務
# ============================================================

print("RoBERTa 訓練任務標籤分布：")

for field in TASK_FIELDS:
    counter = Counter([item.get(field, "MISSING") for item in raw_data])

    print("\n" + "=" * 60)
    print(field)
    print("=" * 60)

    total = sum(counter.values())
    for label, count in counter.items():
        ratio = count / total
        print(f"{label}: {count} 筆 ({ratio:.2%})")

# ============================================================
# 4.2 檢查 GPT-4o + RAG 之後要負責的兩個任務
# ============================================================

RAG_FIELDS = [
    "verification_timeline",
    "evidence_quality"
]

print("\n\nGPT-4o + RAG 任務標籤分布：")

for field in RAG_FIELDS:
    counter = Counter([item.get(field, "MISSING") for item in raw_data])

    print("\n" + "=" * 60)
    print(field)
    print("=" * 60)

    total = sum(counter.values())
    for label, count in counter.items():
        ratio = count / total
        print(f"{label}: {count} 筆 ({ratio:.2%})")

# ============================================================
# 4.3 找出 Misleading 樣本
# ============================================================

misleading_indices = [
    i for i, item in enumerate(raw_data)
    if item.get("evidence_quality") == "Misleading"
]

print("\n" + "=" * 60)
print("Misleading 樣本檢查")
print("=" * 60)
print("Misleading 筆數：", len(misleading_indices))
print("Misleading indices：", misleading_indices)

if len(misleading_indices) > 0:
    idx = misleading_indices[0]
    item = raw_data[idx]

    print("\n唯一 Misleading 樣本內容：")
    print("index:", idx)
    print("id:", item.get("id"))
    print("data:", item.get("data"))
    print("promise_status:", item.get("promise_status"))
    print("verification_timeline:", item.get("verification_timeline"))
    print("evidence_status:", item.get("evidence_status"))
    print("evidence_quality:", item.get("evidence_quality"))
    print("promise_string:", item.get("promise_string"))
    print("evidence_string:", item.get("evidence_string"))

In [ ]:
# ============================================================
# Cell 5. 切分 train / valid / test
# 固定將唯一 Misleading 樣本放入 test_data
# ============================================================

from sklearn.model_selection import train_test_split
from collections import Counter

# ============================================================
# 5.1 分離 Misleading 與非 Misleading 資料
# ============================================================

misleading_data = [
    item for item in raw_data
    if item.get("evidence_quality") == "Misleading"
]

non_misleading_data = [
    item for item in raw_data
    if item.get("evidence_quality") != "Misleading"
]

print("Misleading 筆數：", len(misleading_data))
print("非 Misleading 筆數：", len(non_misleading_data))

if len(misleading_data) != 1:
    raise ValueError("Misleading 筆數不是 1，請先檢查資料。")

misleading_id = misleading_data[0].get("id")
print("固定放入 test_data 的 Misleading id：", misleading_id)

# ============================================================
# 5.2 從 999 筆非 Misleading 中切出 train / valid / test_normal
# ============================================================

# 先切出 99 筆作為普通 test
train_valid_data, test_normal_data = train_test_split(
    non_misleading_data,
    test_size=99,
    random_state=SEED,
    shuffle=True
)

# 剩下 900 筆再切 train 800 / valid 100
train_data, valid_data = train_test_split(
    train_valid_data,
    test_size=100,
    random_state=SEED,
    shuffle=True
)

# test_data = 99 筆普通資料 + 1 筆 Misleading
test_data = test_normal_data + misleading_data

# 打亂 test_data，避免 Misleading 永遠在最後一筆
random.shuffle(test_data)

# ============================================================
# 5.3 檢查切分結果
# ============================================================

print("\n切分結果：")
print("train_data 筆數：", len(train_data))
print("valid_data 筆數：", len(valid_data))
print("test_data 筆數：", len(test_data))

assert len(train_data) == 800
assert len(valid_data) == 100
assert len(test_data) == 100

# 檢查資料是否重複
train_ids = set([item["id"] for item in train_data])
valid_ids = set([item["id"] for item in valid_data])
test_ids = set([item["id"] for item in test_data])

print("\n資料重疊檢查：")
print("train ∩ valid:", len(train_ids & valid_ids))
print("train ∩ test :", len(train_ids & test_ids))
print("valid ∩ test :", len(valid_ids & test_ids))

assert len(train_ids & valid_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(valid_ids & test_ids) == 0

# 檢查 Misleading 是否在 test_data
print("\nMisleading 位置檢查：")
print("Misleading 是否在 train_data:", misleading_id in train_ids)
print("Misleading 是否在 valid_data:", misleading_id in valid_ids)
print("Misleading 是否在 test_data :", misleading_id in test_ids)

assert misleading_id not in train_ids
assert misleading_id not in valid_ids
assert misleading_id in test_ids

# ============================================================
# 5.4 查看各 split 的標籤分布
# ============================================================

def print_split_distribution(name, data_split):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    for field in [
        "promise_status",
        "evidence_status",
        "verification_timeline",
        "evidence_quality"
    ]:
        counter = Counter([item.get(field, "MISSING") for item in data_split])
        total = sum(counter.values())

        print(f"\n{field}:")
        for label, count in counter.items():
            print(f"  {label}: {count} 筆 ({count / total:.2%})")

print_split_distribution("train_data", train_data)
print_split_distribution("valid_data", valid_data)
print_split_distribution("test_data", test_data)

# ============================================================
# 5.5 儲存切分結果
# ============================================================

split_obj = {
    "train_data": train_data,
    "valid_data": valid_data,
    "test_data": test_data,
    "task_fields": TASK_FIELDS,
    "label_lists": LABEL_LISTS,
    "official_weights": OFFICIAL_WEIGHTS,
    "seed": SEED,
    "misleading_id_in_test": misleading_id,
    "note": "The unique Misleading sample is fixed in test_data."
}

with open(SAVE_SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(split_obj, f, ensure_ascii=False, indent=2)

print("\n✅ 已儲存切分資料：", SAVE_SPLIT_PATH)

In [ ]:
# ============================================================
# Cell 6. 建立 Dataset / DataLoader
# ============================================================

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ============================================================
# 6.1 設定模型與 tokenize 參數
# ============================================================

MODEL_NAME = "hfl/chinese-roberta-wwm-ext-large"
MAX_LEN = 512
BATCH_SIZE = 2

print("載入 tokenizer：", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ============================================================
# 6.2 建立 Dataset
# ============================================================

class ESGTask13Dataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = get_text(item)

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        labels = {}

        for field in TASK_FIELDS:
            label = item[field]

            if label not in label2id[field]:
                raise ValueError(
                    f"未知標籤：field={field}, label={label}, id={item.get('id')}"
                )

            labels[field] = torch.tensor(
                label2id[field][label],
                dtype=torch.long
            )

        return {
            "id": item.get("id"),
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

# ============================================================
# 6.3 Collate function
# ============================================================

def collate_fn(batch):
    ids = [x["id"] for x in batch]

    input_ids = torch.stack([x["input_ids"] for x in batch])
    attention_mask = torch.stack([x["attention_mask"] for x in batch])

    labels = {
        field: torch.stack([x["labels"][field] for x in batch])
        for field in TASK_FIELDS
    }

    return {
        "ids": ids,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# ============================================================
# 6.4 建立 Dataset
# ============================================================

train_dataset = ESGTask13Dataset(train_data, tokenizer, MAX_LEN)
valid_dataset = ESGTask13Dataset(valid_data, tokenizer, MAX_LEN)
test_dataset = ESGTask13Dataset(test_data, tokenizer, MAX_LEN)

print("train_dataset 筆數：", len(train_dataset))
print("valid_dataset 筆數：", len(valid_dataset))
print("test_dataset 筆數：", len(test_dataset))

# ============================================================
# 6.5 建立 DataLoader
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("✅ DataLoader 建立完成")

# ============================================================
# 6.6 測試一個 batch
# ============================================================

batch = next(iter(train_loader))

print("\nBatch 測試：")
print("ids:", batch["ids"])
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)

for field in TASK_FIELDS:
    print(f"{field} labels:", batch["labels"][field])
    print(f"{field} labels shape:", batch["labels"][field].shape)

In [ ]:
# ============================================================
# Cell 7. 建立 Single RoBERTa Task 1 + Task 3 模型
# ============================================================

import torch.nn as nn
from transformers import AutoModel

# ============================================================
# 7.1 定義模型
# ============================================================

class SingleRoBERTaTask13(nn.Module):
    def __init__(self, model_name, num_labels_dict, dropout_rate=0.1):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(model_name)

        # 開啟 gradient checkpointing，降低 GPU 記憶體使用量
        self.roberta.gradient_checkpointing_enable()

        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)

        # 只建立 TASK_FIELDS 裡面的分類頭
        self.classifiers = nn.ModuleDict({
            field: nn.Linear(hidden_size, num_labels_dict[field])
            for field in TASK_FIELDS
        })

        # 初始化分類頭
        for classifier in self.classifiers.values():
            nn.init.normal_(classifier.weight, std=0.02)
            nn.init.zeros_(classifier.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # 使用 [CLS] token
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = {
            field: self.classifiers[field](cls_output)
            for field in TASK_FIELDS
        }

        return logits

# ============================================================
# 7.2 建立模型
# ============================================================

model = SingleRoBERTaTask13(
    model_name=MODEL_NAME,
    num_labels_dict=num_labels_dict,
    dropout_rate=0.1
)

model.to(device)

print("✅ 模型建立完成")
print("使用模型：", MODEL_NAME)
print("任務欄位：", TASK_FIELDS)
print("\n分類頭：")
print(model.classifiers)

# ============================================================
# 7.3 測試 forward pass
# ============================================================

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

model.eval()

with torch.no_grad():
    logits = model(input_ids=input_ids, attention_mask=attention_mask)

print("\nForward pass 測試：")
for field in TASK_FIELDS:
    print(f"{field} logits shape:", logits[field].shape)

# 清理 GPU cache
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 8. 設定 Optimizer、Scheduler、Loss
# ============================================================

from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch.nn as nn

# ============================================================
# 8.1 訓練超參數
# ============================================================

EPOCHS = 10
LR = 1e-5
ACCUM_STEPS = 16
MAX_GRAD_NORM = 1.0

# 等效 batch size = BATCH_SIZE * ACCUM_STEPS
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * ACCUM_STEPS

print("訓練超參數：")
print("EPOCHS:", EPOCHS)
print("LR:", LR)
print("BATCH_SIZE:", BATCH_SIZE)
print("ACCUM_STEPS:", ACCUM_STEPS)
print("EFFECTIVE_BATCH_SIZE:", EFFECTIVE_BATCH_SIZE)

# ============================================================
# 8.2 Loss function
# ============================================================

loss_fns = {
    field: nn.CrossEntropyLoss()
    for field in TASK_FIELDS
}

# 根據官方權重，任務 1:任務 3 = 0.20:0.30 = 0.40:0.60
FIELD_WEIGHTS = {
    "promise_status": 0.40,
    "evidence_status": 0.60
}

print("\n訓練 loss 權重：")
print(FIELD_WEIGHTS)

# ============================================================
# 8.3 Optimizer
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LR
)

# ============================================================
# 8.4 Scheduler
# ============================================================

# 注意：使用 ceil，避免 len(train_loader) 不能整除 ACCUM_STEPS 時少算 step
import math

steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_training_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("\nScheduler 設定：")
print("len(train_loader):", len(train_loader))
print("steps_per_epoch:", steps_per_epoch)
print("total_training_steps:", total_training_steps)
print("warmup_steps:", warmup_steps)

# ============================================================
# 8.5 Mixed precision scaler
# ============================================================

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print("\n✅ Cell 8 設定完成")

In [ ]:
# ============================================================
# Cell 9. 評估函數：使用官方權重計算 RoBERTa score
# ============================================================

from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np

# ============================================================
# 9.1 評估函數
# ============================================================

def evaluate(model, dataloader, desc="Evaluating"):
    model.eval()

    all_true = {field: [] for field in TASK_FIELDS}
    all_pred = {field: [] for field in TASK_FIELDS}

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            labels = {
                field: batch["labels"][field].to(device)
                for field in TASK_FIELDS
            }

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                loss = 0.0
                for field in TASK_FIELDS:
                    field_loss = loss_fns[field](
                        logits[field],
                        labels[field]
                    )
                    loss += FIELD_WEIGHTS[field] * field_loss

            total_loss += loss.item()
            total_batches += 1

            for field in TASK_FIELDS:
                preds = torch.argmax(logits[field], dim=1)

                all_true[field].extend(labels[field].detach().cpu().numpy().tolist())
                all_pred[field].extend(preds.detach().cpu().numpy().tolist())

    results = {}

    for field in TASK_FIELDS:
        label_ids = list(range(len(LABEL_LISTS[field])))

        macro_f1 = f1_score(
            all_true[field],
            all_pred[field],
            labels=label_ids,
            average="macro",
            zero_division=0
        )

        micro_f1 = f1_score(
            all_true[field],
            all_pred[field],
            labels=label_ids,
            average="micro",
            zero_division=0
        )

        acc = accuracy_score(
            all_true[field],
            all_pred[field]
        )

        results[field] = {
            "macro_f1": macro_f1,
            "micro_f1": micro_f1,
            "accuracy": acc,
            "y_true": all_true[field],
            "y_pred": all_pred[field]
        }

    # ========================================================
    # 官方權重正規化版分數
    # promise_status 0.20 + evidence_status 0.30 = 0.50
    # ========================================================

    weighted_sum = 0.0
    used_weight_sum = 0.0

    for field in TASK_FIELDS:
        weighted_sum += results[field]["macro_f1"] * OFFICIAL_WEIGHTS[field]
        used_weight_sum += OFFICIAL_WEIGHTS[field]

    roberta_score = weighted_sum / used_weight_sum

    avg_loss = total_loss / max(total_batches, 1)

    return {
        "loss": avg_loss,
        "roberta_score": roberta_score,
        "field_results": results
    }

# ============================================================
# 9.2 印出 classification report
# ============================================================

def print_eval_report(eval_output):
    print(f"Loss: {eval_output['loss']:.4f}")
    print(f"RoBERTa weighted score: {eval_output['roberta_score']:.4f}")

    for field in TASK_FIELDS:
        result = eval_output["field_results"][field]

        label_ids = list(range(len(LABEL_LISTS[field])))
        target_names = LABEL_LISTS[field]

        print("\n" + "=" * 70)
        print(field)
        print("=" * 70)

        print(classification_report(
            result["y_true"],
            result["y_pred"],
            labels=label_ids,
            target_names=target_names,
            zero_division=0
        ))

        print(f"Macro F1: {result['macro_f1']:.4f}")
        print(f"Micro F1: {result['micro_f1']:.4f}")
        print(f"Accuracy: {result['accuracy']:.4f}")

# ============================================================
# 9.3 先測試一次 valid_loader，確認評估函數能跑
# ============================================================

print("開始測試 evaluate()...")

valid_eval_test = evaluate(
    model,
    valid_loader,
    desc="Testing evaluate"
)

print_eval_report(valid_eval_test)

print("\n✅ Cell 9 評估函數建立完成")

In [ ]:
# ============================================================
# Cell 10. 正式訓練 Single RoBERTa Task 1 + Task 3
# ============================================================

best_score = -1.0
best_epoch = -1
history = []

optimizer.zero_grad()

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_train_loss = 0.0
    step_count = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}"
    )

    for step, batch in enumerate(progress_bar, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = {
            field: batch["labels"][field].to(device)
            for field in TASK_FIELDS
        }

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = 0.0

            for field in TASK_FIELDS:
                field_loss = loss_fns[field](
                    logits[field],
                    labels[field]
                )
                loss += FIELD_WEIGHTS[field] * field_loss

            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        total_train_loss += loss.item() * ACCUM_STEPS
        step_count += 1

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad()

        avg_train_loss = total_train_loss / max(step_count, 1)

        progress_bar.set_postfix({
            "loss": f"{avg_train_loss:.4f}"
        })

    avg_train_loss = total_train_loss / max(step_count, 1)

    print("\n" + "=" * 90)
    print(f"Epoch {epoch}/{EPOCHS} 訓練完成")
    print(f"Train Loss: {avg_train_loss:.4f}")

    valid_eval = evaluate(
        model,
        valid_loader,
        desc=f"Valid Epoch {epoch}"
    )

    print_eval_report(valid_eval)

    epoch_record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "valid_loss": valid_eval["loss"],
        "valid_roberta_score": valid_eval["roberta_score"],
        "field_results": {
            field: {
                "macro_f1": valid_eval["field_results"][field]["macro_f1"],
                "micro_f1": valid_eval["field_results"][field]["micro_f1"],
                "accuracy": valid_eval["field_results"][field]["accuracy"]
            }
            for field in TASK_FIELDS
        }
    }

    history.append(epoch_record)

    if valid_eval["roberta_score"] > best_score:
        best_score = valid_eval["roberta_score"]
        best_epoch = epoch

        save_obj = {
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "task_fields": TASK_FIELDS,
            "label_lists": LABEL_LISTS,
            "label2id": label2id,
            "id2label": id2label,
            "num_labels_dict": num_labels_dict,
            "max_len": MAX_LEN,
            "official_weights": OFFICIAL_WEIGHTS,
            "field_weights": FIELD_WEIGHTS,
            "best_score": best_score,
            "best_epoch": best_epoch,
            "seed": SEED
        }

        torch.save(save_obj, SAVE_MODEL_PATH)

        print("\n✅ 儲存新的最佳模型")
        print("SAVE_MODEL_PATH:", SAVE_MODEL_PATH)
        print("Best Epoch:", best_epoch)
        print("Best RoBERTa Score:", f"{best_score:.4f}")

    print("=" * 90)

    # 清理 GPU cache
    torch.cuda.empty_cache()

print("\n訓練完成")
print("Best Epoch:", best_epoch)
print("Best RoBERTa Score:", f"{best_score:.4f}")
print("Best Model Path:", SAVE_MODEL_PATH)

In [ ]:
# ============================================================
# Cell 11. 載入最佳模型並在 test_data 評估
# ============================================================

checkpoint = torch.load(SAVE_MODEL_PATH, map_location=device)

print("載入模型：", SAVE_MODEL_PATH)
print("Best Epoch:", checkpoint["best_epoch"])
print("Best Valid RoBERTa Score:", checkpoint["best_score"])

best_model = SingleRoBERTaTask13(
    model_name=checkpoint["model_name"],
    num_labels_dict=checkpoint["num_labels_dict"],
    dropout_rate=0.1
)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.to(device)
best_model.eval()

print("\n✅ 最佳模型載入完成")

test_eval = evaluate(
    best_model,
    test_loader,
    desc="Testing Best Model"
)

print("\n" + "=" * 90)
print("Best Model on test_data")
print("=" * 90)
print_eval_report(test_eval)

In [ ]:
# ============================================================
# Cell 12. 儲存訓練紀錄與測試結果
# ============================================================

final_training_record = {
    "method": "Single RoBERTa for Task 1 and Task 3",
    "description": "RoBERTa only uses the original data field as input. Task 2 and Task 4 will be handled by GPT-4o + RAG few-shot method.",
    "model_name": MODEL_NAME,
    "save_model_path": SAVE_MODEL_PATH,
    "save_split_path": SAVE_SPLIT_PATH,
    "task_fields": TASK_FIELDS,
    "label_lists": LABEL_LISTS,
    "official_weights": OFFICIAL_WEIGHTS,
    "field_weights": FIELD_WEIGHTS,
    "hyperparameters": {
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "accum_steps": ACCUM_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED
    },
    "best_model": {
        "best_epoch": best_epoch,
        "best_valid_roberta_score": best_score
    },
    "history": history,
    "test_results": {
        "loss": test_eval["loss"],
        "roberta_score": test_eval["roberta_score"],
        "field_results": {
            field: {
                "macro_f1": test_eval["field_results"][field]["macro_f1"],
                "micro_f1": test_eval["field_results"][field]["micro_f1"],
                "accuracy": test_eval["field_results"][field]["accuracy"]
            }
            for field in TASK_FIELDS
        }
    }
}

with open(SAVE_HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(final_training_record, f, ensure_ascii=False, indent=2)

print("✅ 訓練紀錄已儲存：", SAVE_HISTORY_PATH)

print("\n最終摘要：")
print("Best Epoch:", best_epoch)
print("Best Valid RoBERTa Score:", best_score)
print("Test RoBERTa Score:", test_eval["roberta_score"])

for field in TASK_FIELDS:
    print("\n" + field)
    print("Macro F1:", test_eval["field_results"][field]["macro_f1"])
    print("Micro F1:", test_eval["field_results"][field]["micro_f1"])
    print("Accuracy:", test_eval["field_results"][field]["accuracy"])

In [ ]:
# ============================================================
# Hybrid Cell 1. 載入切分資料與設定本機 Hybrid Pipeline 路徑
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置：", device)

if torch.cuda.is_available():
    print("GPU 名稱：", torch.cuda.get_device_name(0))
else:
    print("目前沒有偵測到 GPU，會使用 CPU。")

# ============================================================
# H1.1 本機路徑設定
# ============================================================

BASE_DIR = Path.cwd().resolve()
PROJECT_DIR = BASE_DIR / "outputs"
MODEL_DIR = BASE_DIR / "models"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = PROJECT_DIR / "task13_train_valid_test_split.json"
ROBERTA_MODEL_PATH = MODEL_DIR / "best_single_roberta_task13.pt"

HYBRID_OUTPUT_PATH = PROJECT_DIR / "hybrid_task1234_predictions.json"
HYBRID_SUBMISSION_PATH = PROJECT_DIR / "hybrid_submission.json"

PROJECT_DIR = str(PROJECT_DIR)
MODEL_DIR = str(MODEL_DIR)
SPLIT_PATH = str(SPLIT_PATH)
ROBERTA_MODEL_PATH = str(ROBERTA_MODEL_PATH)
HYBRID_OUTPUT_PATH = str(HYBRID_OUTPUT_PATH)
HYBRID_SUBMISSION_PATH = str(HYBRID_SUBMISSION_PATH)

print("\n路徑設定：")
print("SPLIT_PATH:", SPLIT_PATH)
print("ROBERTA_MODEL_PATH:", ROBERTA_MODEL_PATH)
print("HYBRID_OUTPUT_PATH:", HYBRID_OUTPUT_PATH)
print("HYBRID_SUBMISSION_PATH:", HYBRID_SUBMISSION_PATH)

# ============================================================
# H1.2 檢查檔案是否存在
# ============================================================

if not os.path.exists(SPLIT_PATH):
    raise FileNotFoundError(f"找不到切分資料：{SPLIT_PATH}。請先執行 Cell 1 到 Cell 5 產生 split 檔。")

if not os.path.exists(ROBERTA_MODEL_PATH):
    raise FileNotFoundError(f"找不到 RoBERTa 模型：{ROBERTA_MODEL_PATH}。請先完成訓練，或把模型檔放到 models/。")

print("\nsplit 檔案存在")
print("RoBERTa 模型檔案存在")

# ============================================================
# H1.3 載入切分資料
# ============================================================

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    split_obj = json.load(f)

train_data = split_obj["train_data"]
valid_data = split_obj["valid_data"]
test_data = split_obj["test_data"]

print("\n資料筆數：")
print("train_data:", len(train_data))
print("valid_data:", len(valid_data))
print("test_data:", len(test_data))

# ============================================================
# H1.4 任務與官方權重
# ============================================================

OFFICIAL_WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15
}

ROBERTA_FIELDS = [
    "promise_status",
    "evidence_status"
]

RAG_GPT_FIELDS = [
    "verification_timeline",
    "evidence_quality"
]

ALL_FIELDS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality"
]

LABEL_LISTS_ALL = {
    "promise_status": ["Yes", "No"],
    "verification_timeline": [
        "already",
        "within_2_years",
        "between_2_and_5_years",
        "longer_than_5_years",
        "N/A"
    ],
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"]
}

print("\nHybrid 任務分配：")
print("RoBERTa:", ROBERTA_FIELDS)
print("GPT + RAG:", RAG_GPT_FIELDS)

# ============================================================
# H1.5 文字輸入格式：只使用 data
# ============================================================

def clean_text(value):
    if value is None:
        return ""

    value = str(value).strip()

    if value == "":
        return ""

    if value.lower() == "nan":
        return ""

    return value


def get_text(example):
    """
    只使用官方原始文本 data。
    """
    data_text = clean_text(example.get("data", ""))

    if data_text:
        return data_text

    raise KeyError(f"找不到 data 欄位，目前 keys = {example.keys()}")

print("\n文字範例：")
print(get_text(test_data[0])[:500])

# ============================================================
# H1.6 檢查 Misleading 是否仍在 test_data
# ============================================================

misleading_in_test = [
    item for item in test_data
    if item.get("evidence_quality") == "Misleading"
]

print("\nMisleading in test_data:", len(misleading_in_test))

if len(misleading_in_test) > 0:
    print("Misleading id:", misleading_in_test[0].get("id"))


In [ ]:
# ============================================================
# Hybrid Cell 2. 載入 RoBERTa 並預測 Task 1、Task 3
# ============================================================

import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, f1_score, accuracy_score

# ============================================================
# H2.1 載入 checkpoint
# ============================================================

checkpoint = torch.load(ROBERTA_MODEL_PATH, map_location=device)

MODEL_NAME = checkpoint["model_name"]
TASK_FIELDS = checkpoint["task_fields"]
LABEL_LISTS = checkpoint["label_lists"]
label2id = checkpoint["label2id"]
id2label = checkpoint["id2label"]
num_labels_dict = checkpoint["num_labels_dict"]
MAX_LEN = checkpoint["max_len"]

print("載入 RoBERTa checkpoint：")
print("MODEL_NAME:", MODEL_NAME)
print("TASK_FIELDS:", TASK_FIELDS)
print("Best Epoch:", checkpoint["best_epoch"])
print("Best Valid Score:", checkpoint["best_score"])

# ============================================================
# H2.2 定義模型結構
# 必須跟訓練時一樣
# ============================================================

class SingleRoBERTaTask13(nn.Module):
    def __init__(self, model_name, num_labels_dict, dropout_rate=0.1):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(model_name)
        self.roberta.gradient_checkpointing_enable()

        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)

        self.classifiers = nn.ModuleDict({
            field: nn.Linear(hidden_size, num_labels_dict[field])
            for field in TASK_FIELDS
        })

        for classifier in self.classifiers.values():
            nn.init.normal_(classifier.weight, std=0.02)
            nn.init.zeros_(classifier.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = {
            field: self.classifiers[field](cls_output)
            for field in TASK_FIELDS
        }

        return logits

# ============================================================
# H2.3 載入 tokenizer 與模型
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

roberta_model = SingleRoBERTaTask13(
    model_name=MODEL_NAME,
    num_labels_dict=num_labels_dict,
    dropout_rate=0.1
)

roberta_model.load_state_dict(checkpoint["model_state_dict"])
roberta_model.to(device)
roberta_model.eval()

print("\n✅ RoBERTa 模型載入完成")

# ============================================================
# H2.4 單筆預測函數
# ============================================================

def roberta_predict_one(text):
    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        logits = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    pred = {}

    for field in TASK_FIELDS:
        pred_id = torch.argmax(logits[field], dim=1).item()
        pred[field] = id2label[field][pred_id]

    return pred

# ============================================================
# H2.5 對 test_data 預測 Task 1、Task 3
# ============================================================

roberta_predictions = []

for item in tqdm(test_data, desc="RoBERTa predicting Task 1 & 3"):
    text = get_text(item)
    pred = roberta_predict_one(text)

    roberta_predictions.append({
        "id": item.get("id"),
        "promise_status": pred["promise_status"],
        "evidence_status": pred["evidence_status"]
    })

print("\n✅ RoBERTa 預測完成")
print("預測筆數：", len(roberta_predictions))
print("第一筆預測：")
print(roberta_predictions[0])

# ============================================================
# H2.6 評估 RoBERTa 在 test_data 的 Task 1、Task 3 表現
# ============================================================

print("\n" + "=" * 90)
print("RoBERTa Test Evaluation for Task 1 and Task 3")
print("=" * 90)

roberta_eval_results = {}

for field in TASK_FIELDS:
    y_true = [item[field] for item in test_data]
    y_pred = [pred[field] for pred in roberta_predictions]

    labels = LABEL_LISTS[field]

    print("\n" + "=" * 70)
    print(field)
    print("=" * 70)

    print(classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    ))

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="micro",
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    roberta_eval_results[field] = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "accuracy": acc
    }

    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Accuracy: {acc:.4f}")

roberta_weighted_score = (
    roberta_eval_results["promise_status"]["macro_f1"] * OFFICIAL_WEIGHTS["promise_status"]
    + roberta_eval_results["evidence_status"]["macro_f1"] * OFFICIAL_WEIGHTS["evidence_status"]
) / (
    OFFICIAL_WEIGHTS["promise_status"]
    + OFFICIAL_WEIGHTS["evidence_status"]
)

print("\nRoBERTa weighted score:", roberta_weighted_score)

In [ ]:
# ============================================================
# Hybrid Cell 3. 建立 RAG Retrieval Pool
# 用 train_data + valid_data 作為 few-shot 檢索資料庫
# ============================================================

from collections import Counter

# ============================================================
# H3.1 建立 retrieval pool
# ============================================================

retrieval_source_data = train_data + valid_data

retrieval_pool = []

for item in retrieval_source_data:
    record = {
        "id": item.get("id"),
        "text": get_text(item),
        "verification_timeline": item.get("verification_timeline"),
        "evidence_quality": item.get("evidence_quality"),
        "promise_status": item.get("promise_status"),
        "evidence_status": item.get("evidence_status")
    }

    retrieval_pool.append(record)

print("retrieval_pool 筆數：", len(retrieval_pool))

# ============================================================
# H3.2 檢查 retrieval pool 不包含 Misleading
# ============================================================

retrieval_misleading = [
    item for item in retrieval_pool
    if item.get("evidence_quality") == "Misleading"
]

print("retrieval_pool Misleading 筆數：", len(retrieval_misleading))

if len(retrieval_misleading) > 0:
    print("⚠️ retrieval_pool 含 Misleading，請檢查是否外洩")
    print(retrieval_misleading[0])
else:
    print("✅ retrieval_pool 不含 Misleading，沒有把 test 的 Misleading 放進檢索庫")

# ============================================================
# H3.3 檢查 RAG 任務標籤分布
# ============================================================

print("\nretrieval_pool verification_timeline 分布：")
vt_counter = Counter([item["verification_timeline"] for item in retrieval_pool])
for label, count in vt_counter.items():
    print(f"{label}: {count}")

print("\nretrieval_pool evidence_quality 分布：")
eq_counter = Counter([item["evidence_quality"] for item in retrieval_pool])
for label, count in eq_counter.items():
    print(f"{label}: {count}")

# ============================================================
# H3.4 查看 retrieval 範例
# ============================================================

print("\nretrieval_pool 第一筆範例：")
print("id:", retrieval_pool[0]["id"])
print("verification_timeline:", retrieval_pool[0]["verification_timeline"])
print("evidence_quality:", retrieval_pool[0]["evidence_quality"])
print("text:", retrieval_pool[0]["text"][:500])

In [ ]:
# ============================================================
# Hybrid Cell 4. 建立 Embedding 與 Top-K 檢索函數
# ============================================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ============================================================
# H4.1 載入 Multilingual E5 Embedding 模型
# ============================================================

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

print("載入 embedding model：", EMBEDDING_MODEL_NAME)

embedder = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=str(device)
)

# ============================================================
# H4.2 E5 輸入格式
# E5 建議 passage 用 passage:，query 用 query:
# ============================================================

def e5_passage(text):
    return "passage: " + clean_text(text)

def e5_query(text):
    return "query: " + clean_text(text)

# ============================================================
# H4.3 建立 retrieval embeddings
# ============================================================

retrieval_texts = [
    e5_passage(item["text"])
    for item in retrieval_pool
]

print("開始建立 retrieval embeddings...")
print("retrieval_texts 筆數：", len(retrieval_texts))

retrieval_embeddings = embedder.encode(
    retrieval_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("✅ retrieval embeddings 建立完成")
print("retrieval_embeddings shape:", retrieval_embeddings.shape)

# ============================================================
# H4.4 Top-K 檢索函數
# ============================================================

TOP_K = 6

def retrieve_top_k_examples(query_text, top_k=TOP_K):
    query_embedding = embedder.encode(
        [e5_query(query_text)],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        retrieval_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    examples = []

    for idx in top_indices:
        example = retrieval_pool[idx].copy()
        example["similarity_score"] = float(scores[idx])
        examples.append(example)

    return examples

# ============================================================
# H4.5 測試檢索一筆 test_data
# ============================================================

sample_item = test_data[0]
sample_text = get_text(sample_item)

sample_examples = retrieve_top_k_examples(sample_text, top_k=TOP_K)

print("\n測試檢索：")
print("test id:", sample_item.get("id"))
print("test verification_timeline:", sample_item.get("verification_timeline"))
print("test evidence_quality:", sample_item.get("evidence_quality"))
print("test text:", sample_text[:300])

for i, ex in enumerate(sample_examples, 1):
    print("\n" + "-" * 80)
    print(f"Top {i} | score = {ex['similarity_score']:.4f}")
    print("id:", ex["id"])
    print("verification_timeline:", ex["verification_timeline"])
    print("evidence_quality:", ex["evidence_quality"])
    print("text:", ex["text"][:300])

In [ ]:
# ============================================================
# Hybrid Cell 11. 建立雙 Prompt v3
# Timeline prompt v3 + Quality prompt v3
# ============================================================

# ============================================================
# 11.1 合法標籤
# ============================================================

VALID_VERIFICATION_TIMELINE = [
    "already",
    "within_2_years",
    "between_2_and_5_years",
    "longer_than_5_years",
    "N/A"
]

VALID_EVIDENCE_QUALITY = [
    "Clear",
    "Not Clear",
    "Misleading",
    "N/A"
]

# ============================================================
# 11.2 確認 RoBERTa prediction by id
# ============================================================

roberta_pred_by_id = {
    item["id"]: item
    for item in roberta_predictions
}

print("RoBERTa predictions 筆數：", len(roberta_pred_by_id))

# ============================================================
# 11.3 Timeline 專用 Prompt v3
# ============================================================

def build_timeline_prompt_v3(test_text, retrieved_examples, roberta_pred):
    """
    Timeline prompt v3:
    只預測 verification_timeline。
    不預測 evidence_quality。
    """

    examples_text = ""

    for i, ex in enumerate(retrieved_examples, 1):
        examples_text += f"""
【相似範例 {i}】
文本：
{ex["text"]}

標籤：
verification_timeline = {ex["verification_timeline"]}
promise_status = {ex["promise_status"]}
evidence_status = {ex["evidence_status"]}
"""

    prompt = f"""
你是一位嚴謹的 ESG Promise Verification 標註專家。
請根據測試文本、RoBERTa 輔助預測、標籤定義與相似範例，只判斷 verification_timeline。

你只需要預測：
1. verification_timeline

請不要預測 promise_status。
請不要預測 evidence_status。
請不要預測 evidence_quality。
請只輸出 JSON。

============================================================
RoBERTa 輔助預測
============================================================

promise_status = {roberta_pred["promise_status"]}
evidence_status = {roberta_pred["evidence_status"]}

注意：
- RoBERTa 預測只是輔助，不一定完全正確。
- verification_timeline 主要看「承諾或行動的時間狀態」。
- 不要只因為文本出現 2024 年就一定判 already。
- 不要只因為文本出現「持續」就一定判 already。
- 要判斷文本重點是在「已完成事實」還是「未來承諾/後續目標」。

============================================================
verification_timeline 標籤定義
============================================================

- already：
文本主要描述 ESG 行動、成果、制度、活動、認證、揭露、調查、稽核或數據已經發生、已經存在、已經完成。
常見線索：
「已完成」、「已取得」、「已通過」、「已達成」、「已執行」、「已設立」、
「於 2024 年完成」、「2024 年共...」、「累計...」、「已累積...」、「自某年起已持續...」。

- within_2_years：
文本明確表示某項承諾、目標或行動會在 2 年內完成。
只有明確短期期限時才使用，例如：
「一年內」、「兩年內」、「2025 年前」、「2026 年前」。
如果沒有明確短期完成期限，不要亂判 within_2_years。

- between_2_and_5_years：
文本描述未來承諾、後續追蹤、逐步推動、持續改善、持續優化、階段性推動，且不像是超長期目標。
常見線索：
「將持續」、「後續追蹤」、「持續改善」、「持續優化」、「持續精進」、
「逐步建立」、「逐步推動」、「規劃」、「訂定目標」、「階段性推動」、
「期待」、「期盼」、「共同推動」、「未來將...」。
注意：即使文本前面有已完成成果，只要重點落在後續追蹤或未來推動，也可能是 between_2_and_5_years。

- longer_than_5_years：
文本明確描述長期承諾或長期目標。
常見線索：
「長期目標」、「中長期目標」、「短中長期目標」、
「2030」、「2040」、「2050」、「淨零」、「碳中和」。
注意：「長年」、「長期合作」如果只是描述過去長期合作，不一定是 longer_than_5_years，可能是 already。

- N/A：
文本沒有明確時間資訊，或只是描述治理架構、組織職責、委員會、管理流程、政策原則，沒有可判斷的承諾時間。
常見線索：
「委員會」、「執行小組」、「治理架構」、「管理架構」、「政策制定」、
「統籌推動」、「負責執行」、「最高督導單位」。
但如果這些文本同時有明確已完成成果或未來目標，仍要依內容判斷 already / future label。

============================================================
重要判斷規則 v3
============================================================

1. 先判斷文本重點：
   - 主要是已發生事實 → already
   - 主要是未來目標、後續追蹤、階段性推動 → between_2_and_5_years 或 longer_than_5_years
   - 主要是治理架構或職責描述，沒有時間承諾 → N/A

2. 如果同時有「已完成事實」和「未來持續追蹤」：
   - 若後續追蹤/未來推動是文本重點，偏 between_2_and_5_years。
   - 若未來只是附帶描述，主體是已完成成果，偏 already。

3. 「2030 / 2040 / 2050 / 淨零 / 碳中和」通常表示 longer_than_5_years。
4. 「短、中、長期目標」通常表示 longer_than_5_years。
5. 「長年合作」「長期合作」描述過去合作關係時，不等於 longer_than_5_years。
6. 沒有明確時間或承諾時間時，不要硬猜，判 N/A。

============================================================
相似範例
============================================================

{examples_text}

============================================================
測試文本
============================================================

{test_text}

============================================================
輸出格式
============================================================

請只輸出以下 JSON 格式，不要加入 markdown，不要加入說明文字：

{{
  "verification_timeline": "already"
}}
"""

    return prompt

# ============================================================
# 11.4 Evidence Quality 專用 Prompt v3
# ============================================================

def build_quality_prompt_v3(test_text, retrieved_examples, roberta_pred):
    """
    Quality prompt v3:
    只預測 evidence_quality。
    不預測 verification_timeline。
    """

    examples_text = ""

    for i, ex in enumerate(retrieved_examples, 1):
        examples_text += f"""
【相似範例 {i}】
文本：
{ex["text"]}

標籤：
promise_status = {ex["promise_status"]}
evidence_status = {ex["evidence_status"]}
evidence_quality = {ex["evidence_quality"]}
"""

    prompt = f"""
你是一位嚴謹的 ESG Promise Verification 標註專家。
請根據測試文本、RoBERTa 輔助預測、標籤定義與相似範例，只判斷 evidence_quality。

你只需要預測：
1. evidence_quality

請不要預測 verification_timeline。
請不要預測 promise_status。
請不要預測 evidence_status。
請只輸出 JSON。

============================================================
RoBERTa 輔助預測
============================================================

promise_status = {roberta_pred["promise_status"]}
evidence_status = {roberta_pred["evidence_status"]}

注意：
- RoBERTa 預測只是輔助，不一定完全正確。
- evidence_status 對 evidence_quality 很重要。
- 如果 evidence_status = N/A，通常 evidence_quality = N/A。
- 如果 evidence_status = No，通常 evidence_quality = N/A。
- 但如果文本明顯使用 ESG 正面語氣、治理語氣或承諾語氣，卻沒有具體證據，可能是 Not Clear 或 Misleading。

============================================================
evidence_quality 標籤定義
============================================================

- Clear：
文本提供具體、可驗證的證據。
常見證據包括：
明確年份、金額、百分比、數量、人次、件數、場次、第三方驗證、ISO 認證、SBTi、獎項、具體成果、量化指標、已完成成果、達成率、改善率、投入金額、捐助金額。

- Not Clear：
文本有 ESG 承諾、方向、政策、制度、行動或管理描述，但證據不足、不夠具體。
常見情況：
只有「致力於」、「持續推動」、「強化」、「提升」、「落實」、「重視」、「改善」、「遵循」等方向性語句，
但缺少可驗證數據、具體成果、明確認證、量化指標。

- Misleading：
文本使用很正式、正面、承諾式、治理式或社會責任語氣，使讀者可能誤以為公司已有明確 ESG 作為、證據或驗證，
但實際上沒有具體數據、沒有驗證依據、沒有明確成果、沒有可查證證據。
Misleading 比 Not Clear 更像「過度包裝」或「容易讓人誤解已有證據」。
常見語氣：
「致力於穩健經營」、「確保持續成長」、「兼顧股東權益、員工發展與社會責任」、
「提升企業競爭力」、「承諾遵循法規」、「維持高標準透明度與公司治理」。
如果整段文字多是漂亮口號與治理承諾，但沒有具體數字、年份、認證、成果，請考慮 Misleading。

- N/A：
文本不適用 evidence_quality 判斷。
常見情況：
沒有 evidence、沒有 promise，或主要是背景介紹、組織架構、管理職責、政策描述，且不構成可評估的證據品質。

============================================================
重要判斷規則 v3
============================================================

1. 如果 evidence_status = Yes：
   - 有具體數據、年份、金額、百分比、認證、獎項、量化成果 → Clear
   - 只有方向性描述或制度性描述，缺乏可驗證成果 → Not Clear
   - 語氣很正面、很像 ESG 成果，但沒有具體證據 → Misleading

2. 如果 evidence_status = No 或 N/A：
   - 通常 evidence_quality = N/A
   - 但如果文本有強烈 ESG 口號式承諾、治理包裝、社會責任語氣，且缺乏證據 → Misleading 或 Not Clear

3. 不要因為文本有「2024 年」就自動判 Clear。
   要看 2024 年是否連結到具體成果、數據、認證、活動或量化證據。

4. 不要因為文本很正式、很像永續報告就自動判 Clear。
   如果沒有具體可驗證內容，應判 Not Clear 或 Misleading。

5. 如果文本有明確量化證據，例如「120 萬元」、「228 位」、「93.4%」、「1.74 億元」、「ISO 27001」、「SBTi 通過」，通常判 Clear。

6. Misleading 只在文本明顯有過度正面包裝、承諾式語氣，且缺乏具體證據時使用。
   如果只是一般證據不足，判 Not Clear。

============================================================
相似範例
============================================================

{examples_text}

============================================================
測試文本
============================================================

{test_text}

============================================================
輸出格式
============================================================

請只輸出以下 JSON 格式，不要加入 markdown，不要加入說明文字：

{{
  "evidence_quality": "Clear"
}}
"""

    return prompt

# ============================================================
# 11.5 測試建立 prompt
# ============================================================

sample_item = test_data[0]
sample_text = get_text(sample_item)
sample_examples = retrieve_top_k_examples(sample_text, top_k=TOP_K)
sample_roberta_pred = roberta_pred_by_id[sample_item["id"]]

sample_timeline_prompt_v3 = build_timeline_prompt_v3(
    test_text=sample_text,
    retrieved_examples=sample_examples,
    roberta_pred=sample_roberta_pred
)

sample_quality_prompt_v3 = build_quality_prompt_v3(
    test_text=sample_text,
    retrieved_examples=sample_examples,
    roberta_pred=sample_roberta_pred
)

print("Timeline Prompt v3 長度:", len(sample_timeline_prompt_v3))
print("Quality Prompt v3 長度:", len(sample_quality_prompt_v3))

print("\nTimeline Prompt v3 前 2000 字：")
print(sample_timeline_prompt_v3[:2000])

print("\n" + "=" * 80)
print("Quality Prompt v3 前 2000 字：")
print(sample_quality_prompt_v3[:2000])

In [ ]:
# ============================================================
# Hybrid Cell 13. Prompt v3 全 100 筆可續跑版
# Timeline Prompt v3 + Quality Prompt v3
# ============================================================

import os
import json
import time
from tqdm import tqdm
from openai import OpenAI
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    api_key = getpass("請貼上你的 OpenAI API Key：").strip()
    os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GPT_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o")
GPT_TEMPERATURE = 0

print("✅ OpenAI client 已建立")
print("GPT_MODEL:", GPT_MODEL)
print("GPT_TEMPERATURE:", GPT_TEMPERATURE)
# ============================================================
# 13.1 設定進度儲存路徑
# ============================================================

TIMELINE_V3_PROGRESS_PATH = os.path.join(
    PROJECT_DIR,
    "gpt_timeline_v3_progress.json"
)

QUALITY_V3_PROGRESS_PATH = os.path.join(
    PROJECT_DIR,
    "gpt_quality_v3_progress.json"
)

print("TIMELINE_V3_PROGRESS_PATH:", TIMELINE_V3_PROGRESS_PATH)
print("QUALITY_V3_PROGRESS_PATH:", QUALITY_V3_PROGRESS_PATH)

# ============================================================
# 13.2 讀取既有進度
# ============================================================

if os.path.exists(TIMELINE_V3_PROGRESS_PATH):
    with open(TIMELINE_V3_PROGRESS_PATH, "r", encoding="utf-8") as f:
        timeline_v3_predictions = json.load(f)
    print("✅ 已讀取既有 timeline v3 預測進度：", len(timeline_v3_predictions))
else:
    timeline_v3_predictions = []
    print("尚無 timeline v3 進度，從 0 開始")

if os.path.exists(QUALITY_V3_PROGRESS_PATH):
    with open(QUALITY_V3_PROGRESS_PATH, "r", encoding="utf-8") as f:
        quality_v3_predictions = json.load(f)
    print("✅ 已讀取既有 quality v3 預測進度：", len(quality_v3_predictions))
else:
    quality_v3_predictions = []
    print("尚無 quality v3 進度，從 0 開始")

timeline_completed_ids = set([item["id"] for item in timeline_v3_predictions])
quality_completed_ids = set([item["id"] for item in quality_v3_predictions])

print("timeline 已完成 id 數量：", len(timeline_completed_ids))
print("quality 已完成 id 數量：", len(quality_completed_ids))
print("總 test_data 筆數：", len(test_data))

# ============================================================
# 13.3 單筆 Timeline v3 預測函數
# ============================================================

def gpt_predict_timeline_v3_full(test_item, top_k=TOP_K, max_retries=5):
    item_id = test_item["id"]
    test_text = get_text(test_item)

    retrieved_examples = retrieve_top_k_examples(test_text, top_k=top_k)
    roberta_pred = roberta_pred_by_id[item_id]

    prompt = build_timeline_prompt_v3(
        test_text=test_text,
        retrieved_examples=retrieved_examples,
        roberta_pred=roberta_pred
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "你是一位嚴謹的 ESG Promise Verification 標註專家。你只能輸出合法 JSON。"
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=GPT_TEMPERATURE
            )

            content = response.choices[0].message.content
            parsed = safe_parse_json(content)
            normalized = normalize_timeline_label(parsed)

            return {
                "id": item_id,
                "verification_timeline": normalized["verification_timeline"],
                "api_success": True,
                "error": None,
                "raw_response": content,
                "roberta_promise_status": roberta_pred["promise_status"],
                "roberta_evidence_status": roberta_pred["evidence_status"]
            }

        except Exception as e:
            last_error = str(e)
            print(f"id={item_id} Timeline v3 第 {attempt} 次失敗：{last_error}")

            wait_time = 10 * attempt
            print(f"等待 {wait_time} 秒後重試...")
            time.sleep(wait_time)

    return {
        "id": item_id,
        "verification_timeline": "N/A",
        "api_success": False,
        "error": last_error,
        "raw_response": None,
        "roberta_promise_status": roberta_pred_by_id[item_id]["promise_status"],
        "roberta_evidence_status": roberta_pred_by_id[item_id]["evidence_status"]
    }

# ============================================================
# 13.4 單筆 Quality v3 預測函數
# ============================================================

def gpt_predict_quality_v3_full(test_item, top_k=TOP_K, max_retries=5):
    item_id = test_item["id"]
    test_text = get_text(test_item)

    retrieved_examples = retrieve_top_k_examples(test_text, top_k=top_k)
    roberta_pred = roberta_pred_by_id[item_id]

    prompt = build_quality_prompt_v3(
        test_text=test_text,
        retrieved_examples=retrieved_examples,
        roberta_pred=roberta_pred
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "你是一位嚴謹的 ESG Promise Verification 標註專家。你只能輸出合法 JSON。"
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=GPT_TEMPERATURE
            )

            content = response.choices[0].message.content
            parsed = safe_parse_json(content)
            normalized = normalize_quality_label(parsed)

            return {
                "id": item_id,
                "evidence_quality": normalized["evidence_quality"],
                "api_success": True,
                "error": None,
                "raw_response": content,
                "roberta_promise_status": roberta_pred["promise_status"],
                "roberta_evidence_status": roberta_pred["evidence_status"]
            }

        except Exception as e:
            last_error = str(e)
            print(f"id={item_id} Quality v3 第 {attempt} 次失敗：{last_error}")

            wait_time = 10 * attempt
            print(f"等待 {wait_time} 秒後重試...")
            time.sleep(wait_time)

    return {
        "id": item_id,
        "evidence_quality": "N/A",
        "api_success": False,
        "error": last_error,
        "raw_response": None,
        "roberta_promise_status": roberta_pred_by_id[item_id]["promise_status"],
        "roberta_evidence_status": roberta_pred_by_id[item_id]["evidence_status"]
    }

# ============================================================
# 13.5 正式可續跑預測
# ============================================================

for idx, item in enumerate(tqdm(test_data, desc="Prompt v3 full predicting")):
    item_id = item["id"]

    print("\n" + "=" * 100)
    print(f"index {idx} | id {item_id}")

    # ------------------------------------------------------------
    # Timeline v3
    # ------------------------------------------------------------
    if item_id in timeline_completed_ids:
        print("Timeline v3 已完成，跳過")
    else:
        timeline_pred = gpt_predict_timeline_v3_full(item)
        timeline_v3_predictions.append(timeline_pred)
        timeline_completed_ids.add(item_id)

        with open(TIMELINE_V3_PROGRESS_PATH, "w", encoding="utf-8") as f:
            json.dump(timeline_v3_predictions, f, ensure_ascii=False, indent=2)

        print("[Timeline]")
        print("Gold:", item["verification_timeline"])
        print("Pred:", timeline_pred["verification_timeline"])
        print("API success:", timeline_pred["api_success"])

        time.sleep(3)

    # ------------------------------------------------------------
    # Quality v3
    # ------------------------------------------------------------
    if item_id in quality_completed_ids:
        print("Quality v3 已完成，跳過")
    else:
        quality_pred = gpt_predict_quality_v3_full(item)
        quality_v3_predictions.append(quality_pred)
        quality_completed_ids.add(item_id)

        with open(QUALITY_V3_PROGRESS_PATH, "w", encoding="utf-8") as f:
            json.dump(quality_v3_predictions, f, ensure_ascii=False, indent=2)

        print("[Evidence Quality]")
        print("Gold:", item["evidence_quality"])
        print("Pred:", quality_pred["evidence_quality"])
        print("API success:", quality_pred["api_success"])

        time.sleep(5)

    print(
        f"目前進度：Timeline {len(timeline_v3_predictions)}/{len(test_data)} | "
        f"Quality {len(quality_v3_predictions)}/{len(test_data)}"
    )

print("\n✅ Prompt v3 全部預測完成")
print("Timeline 預測筆數：", len(timeline_v3_predictions))
print("Timeline API success：", sum(1 for x in timeline_v3_predictions if x["api_success"]))
print("Timeline API failed：", sum(1 for x in timeline_v3_predictions if not x["api_success"]))

print("\nQuality 預測筆數：", len(quality_v3_predictions))
print("Quality API success：", sum(1 for x in quality_v3_predictions if x["api_success"]))
print("Quality API failed：", sum(1 for x in quality_v3_predictions if not x["api_success"]))

print("\n進度檔案：")
print("Timeline:", TIMELINE_V3_PROGRESS_PATH)
print("Quality :", QUALITY_V3_PROGRESS_PATH)

In [ ]:
# ============================================================
# Hybrid Cell 16. Timeline Postprocess v4
# 不呼叫 GPT，只修正 GPT Timeline Prompt v3 的 already 過度預測
# ============================================================

import re
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score

# ============================================================
# 16.1 Timeline 後處理規則
# ============================================================

def timeline_postprocess_v4(item, pred_vt):
    """
    修正 GPT Timeline Prompt v3 的常見錯誤：
    1. longer_than_5_years 被誤判成 already
    2. N/A 被誤判成 already
    3. between_2_and_5_years 被誤判成 already
    """
    text = clean_text(get_text(item))
    item_id = item["id"]

    # ------------------------------------------------------------
    # A. 明確長期目標 / 長期承諾
    # ------------------------------------------------------------
    long_term_patterns = [
        "2050",
        "2040",
        "2035",
        "2030",
        "淨零",
        "碳中和",
        "長期目標",
        "中長期目標",
        "短中長期目標",
        "短、中、長期目標",
        "長期策略",
        "長期減碳",
        "多年計畫",
        "8 年",
        "八年",
        "五年以上",
        "超過五年",
        "50 多年",
        "五十多年"
    ]

    if any(k in text for k in long_term_patterns):
        # 但「長期合作」「長年推動」若明顯是過去已發生成果，不一定是 longer_than_5_years
        past_activity_clues = [
            "2024 年共",
            "2024年共",
            "共捐助",
            "贊助",
            "捐助",
            "已累積",
            "累積",
            "自 2017 年起",
            "自 2018 年起",
            "自 2021 年起",
            "活動",
            "賽事"
        ]

        if not any(k in text for k in past_activity_clues):
            return "longer_than_5_years"

    # ------------------------------------------------------------
    # B. 明確 N/A 類：風險表、背景描述、課程/架構/委員會/評鑑流程
    # ------------------------------------------------------------
    na_patterns = [
        "風險報告",
        "全球風險報告",
        "風險管理概論",
        "風險管理框架",
        "新興風險識別",
        "評鑑小組",
        "年度評鑑",
        "審計委員會",
        "董事會制訂",
        "委員會成員",
        "生命週期角度檢視",
        "背景下",
        "依據 ENCORE",
        "生物多樣性熱點影響分析",
        "熱區",
        "不適用",
        "自身營運",
        "負面新聞 / 訴訟",
        "裁罰風險",
        "課程內容涵蓋",
        "培訓課程",
        "組織架構",
        "管理架構",
        "治理架構"
    ]

    if any(k in text for k in na_patterns):
        return "N/A"

    # 如果 RoBERTa 預測 promise_status = No 且 evidence_status = N/A，timeline 通常不適用
    roberta_pred = roberta_pred_by_id[item_id]
    if roberta_pred["promise_status"] == "No" and roberta_pred["evidence_status"] == "N/A":
        # 但如果文本明確有長期目標，前面已經先處理
        return "N/A"

    # ------------------------------------------------------------
    # C. 中期承諾 / 後續推動
    # ------------------------------------------------------------
    mid_term_patterns = [
        "將持續",
        "後續追蹤",
        "持續追蹤",
        "持續改善",
        "持續優化",
        "持續精進",
        "逐步建立",
        "逐步推動",
        "規劃",
        "計畫與",
        "建構",
        "未來將",
        "期待",
        "期盼",
        "共同推動",
        "共同努力",
        "強化",
        "促進人才",
        "擴展高潛力",
        "培育",
        "建立永續夥伴關係",
        "因應減碳",
        "永續潮流",
        "法定報告",
        "揭露永續相關資訊"
    ]

    if any(k in text for k in mid_term_patterns):
        # 若有 2030/2050 等，應該給 longer_than_5_years，已在 A 處理
        return "between_2_and_5_years"

    # ------------------------------------------------------------
    # D. 明確 already 類：具體過去成果 / 活動 / 數據
    # ------------------------------------------------------------
    already_patterns = [
        "已完成",
        "已取得",
        "已通過",
        "已達成",
        "已執行",
        "已設立",
        "2024 年共",
        "2024年共",
        "共捐助",
        "共投入",
        "累計",
        "已累積",
        "贊助",
        "捐助",
        "獲頒",
        "完成訓練",
        "每季召開",
        "通過考試",
        "取得認證",
        "通過認證",
        "導入 ISO",
        "ISO 45001",
        "ISO 27001",
        "ISO 14064"
    ]

    if any(k in text for k in already_patterns):
        return "already"

    # ------------------------------------------------------------
    # E. 其他：保留 GPT 原本預測
    # ------------------------------------------------------------
    return pred_vt


# ============================================================
# 16.2 套用 postprocess 到全部 test_data
# ============================================================

timeline_v4_predictions = []

for item in test_data:
    item_id = item["id"]
    pred_vt_before = timeline_v3_pred_by_id[item_id]["verification_timeline"]
    pred_vt_after = timeline_postprocess_v4(item, pred_vt_before)

    timeline_v4_predictions.append({
        "id": item_id,
        "verification_timeline": pred_vt_after,
        "before_verification_timeline": pred_vt_before
    })

timeline_v4_pred_by_id = {
    item["id"]: item
    for item in timeline_v4_predictions
}

print("timeline_v4_predictions 筆數：", len(timeline_v4_predictions))

# ============================================================
# 16.3 評估 v3 before / v4 after
# ============================================================

labels_vt = LABEL_LISTS_ALL["verification_timeline"]

y_true_vt = [item["verification_timeline"] for item in test_data]

y_pred_vt_v3 = [
    timeline_v3_pred_by_id[item["id"]]["verification_timeline"]
    for item in test_data
]

y_pred_vt_v4 = [
    timeline_v4_pred_by_id[item["id"]]["verification_timeline"]
    for item in test_data
]

print("\n" + "=" * 90)
print("Timeline v3 Before Postprocess")
print("=" * 90)

v3_macro = f1_score(
    y_true_vt,
    y_pred_vt_v3,
    labels=labels_vt,
    average="macro",
    zero_division=0
)
v3_acc = accuracy_score(y_true_vt, y_pred_vt_v3)

print("v3 Macro F1:", v3_macro)
print("v3 Accuracy:", v3_acc)

print("\n" + "=" * 90)
print("Timeline v4 After Postprocess")
print("=" * 90)

print(classification_report(
    y_true_vt,
    y_pred_vt_v4,
    labels=labels_vt,
    zero_division=0
))

vt_v4_macro_f1 = f1_score(
    y_true_vt,
    y_pred_vt_v4,
    labels=labels_vt,
    average="macro",
    zero_division=0
)

vt_v4_micro_f1 = f1_score(
    y_true_vt,
    y_pred_vt_v4,
    labels=labels_vt,
    average="micro",
    zero_division=0
)

vt_v4_acc = accuracy_score(y_true_vt, y_pred_vt_v4)

print(f"v4 Macro F1: {vt_v4_macro_f1:.4f}")
print(f"v4 Micro F1: {vt_v4_micro_f1:.4f}")
print(f"v4 Accuracy: {vt_v4_acc:.4f}")

# ============================================================
# 16.4 查看被修改案例
# ============================================================

changed_rows = []

for item in test_data:
    item_id = item["id"]
    before = timeline_v3_pred_by_id[item_id]["verification_timeline"]
    after = timeline_v4_pred_by_id[item_id]["verification_timeline"]

    if before != after:
        changed_rows.append({
            "id": item_id,
            "gold_vt": item["verification_timeline"],
            "before_vt": before,
            "after_vt": after,
            "before_correct": item["verification_timeline"] == before,
            "after_correct": item["verification_timeline"] == after,
            "text": get_text(item)[:300]
        })

changed_timeline_v4_df = pd.DataFrame(changed_rows)

print("\n被 postprocess 修改的案例數：", len(changed_timeline_v4_df))
display(changed_timeline_v4_df.head(60))

# ============================================================
# 16.5 錯誤案例數比較
# ============================================================

before_errors = sum(
    1 for item in test_data
    if item["verification_timeline"] != timeline_v3_pred_by_id[item["id"]]["verification_timeline"]
)

after_errors = sum(
    1 for item in test_data
    if item["verification_timeline"] != timeline_v4_pred_by_id[item["id"]]["verification_timeline"]
)

print("\n錯誤數比較：")
print("Before errors:", before_errors)
print("After errors :", after_errors)
print("改善筆數：", before_errors - after_errors)

In [ ]:
# ============================================================
# Hybrid Cell 17. 合併 Hybrid v4 並計算官方加權分數
# Task 1 promise_status        = RoBERTa
# Task 2 verification_timeline = GPT Timeline v3 + Postprocess v4
# Task 3 evidence_status       = RoBERTa
# Task 4 evidence_quality      = GPT Quality Prompt v3
# ============================================================

import os
import json
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score

# ============================================================
# 17.1 建立 prediction by id
# ============================================================

roberta_pred_by_id = {
    item["id"]: item
    for item in roberta_predictions
}

timeline_v4_pred_by_id = {
    item["id"]: item
    for item in timeline_v4_predictions
}

quality_v3_pred_by_id = {
    item["id"]: item
    for item in quality_v3_predictions
}

# 檢查缺漏
missing_roberta = [
    item["id"] for item in test_data
    if item["id"] not in roberta_pred_by_id
]

missing_timeline_v4 = [
    item["id"] for item in test_data
    if item["id"] not in timeline_v4_pred_by_id
]

missing_quality_v3 = [
    item["id"] for item in test_data
    if item["id"] not in quality_v3_pred_by_id
]

print("missing_roberta:", missing_roberta)
print("missing_timeline_v4:", missing_timeline_v4)
print("missing_quality_v3:", missing_quality_v3)

if missing_roberta or missing_timeline_v4 or missing_quality_v3:
    raise ValueError("有任務缺少預測，請先補齊。")

# ============================================================
# 17.2 合併四任務預測
# ============================================================

hybrid_v4_predictions = []

for item in test_data:
    item_id = item["id"]

    pred = {
        "id": item_id,
        "promise_status": roberta_pred_by_id[item_id]["promise_status"],
        "verification_timeline": timeline_v4_pred_by_id[item_id]["verification_timeline"],
        "evidence_status": roberta_pred_by_id[item_id]["evidence_status"],
        "evidence_quality": quality_v3_pred_by_id[item_id]["evidence_quality"]
    }

    hybrid_v4_predictions.append(pred)

print("hybrid_v4_predictions 筆數：", len(hybrid_v4_predictions))
print("第一筆 hybrid_v4 prediction:")
print(hybrid_v4_predictions[0])

hybrid_v4_pred_by_id = {
    item["id"]: item
    for item in hybrid_v4_predictions
}

# ============================================================
# 17.3 評估四個任務
# ============================================================

hybrid_v4_eval_results = {}

for field in ALL_FIELDS:
    y_true = [
        item[field]
        for item in test_data
    ]

    y_pred = [
        hybrid_v4_pred_by_id[item["id"]][field]
        for item in test_data
    ]

    labels = LABEL_LISTS_ALL[field]

    print("\n" + "=" * 90)
    print(f"Hybrid v4 Evaluation: {field}")
    print("=" * 90)

    print(classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    ))

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="micro",
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    hybrid_v4_eval_results[field] = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "accuracy": acc
    }

    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Accuracy: {acc:.4f}")

# ============================================================
# 17.4 官方加權總分
# ============================================================

official_score_v4 = (
    hybrid_v4_eval_results["promise_status"]["macro_f1"] * OFFICIAL_WEIGHTS["promise_status"]
    + hybrid_v4_eval_results["evidence_status"]["macro_f1"] * OFFICIAL_WEIGHTS["evidence_status"]
    + hybrid_v4_eval_results["evidence_quality"]["macro_f1"] * OFFICIAL_WEIGHTS["evidence_quality"]
    + hybrid_v4_eval_results["verification_timeline"]["macro_f1"] * OFFICIAL_WEIGHTS["verification_timeline"]
)

print("\n" + "=" * 90)
print("Hybrid v4 Official Weighted Score")
print("=" * 90)

print("promise_status Macro F1:", hybrid_v4_eval_results["promise_status"]["macro_f1"])
print("verification_timeline Macro F1:", hybrid_v4_eval_results["verification_timeline"]["macro_f1"])
print("evidence_status Macro F1:", hybrid_v4_eval_results["evidence_status"]["macro_f1"])
print("evidence_quality Macro F1:", hybrid_v4_eval_results["evidence_quality"]["macro_f1"])

print("\nOfficial Weighted Score v4:", official_score_v4)

# ============================================================
# 17.5 與 v2 / v3 比較
# ============================================================

print("\n" + "=" * 90)
print("Hybrid Score Comparison")
print("=" * 90)

try:
    print("Hybrid v2 Official Score:", official_score)
except:
    print("Hybrid v2 Official Score: 找不到 official_score 變數")

try:
    print("Hybrid v3 Official Score:", official_score_v3)
except:
    print("Hybrid v3 Official Score: 找不到 official_score_v3 變數")

print("Hybrid v4 Official Score:", official_score_v4)

# ============================================================
# 17.6 儲存 Hybrid v4 結果
# ============================================================

HYBRID_V4_EVAL_PATH = os.path.join(
    PROJECT_DIR,
    "hybrid_v4_eval_results.json"
)

HYBRID_V4_PRED_PATH = os.path.join(
    PROJECT_DIR,
    "hybrid_v4_predictions.json"
)

save_obj = {
    "method": "Hybrid v4: RoBERTa Task1/3 + GPT Timeline v3 + Timeline Postprocess v4 + GPT Quality v3",
    "official_weights": OFFICIAL_WEIGHTS,
    "hybrid_predictions": hybrid_v4_predictions,
    "hybrid_eval_results": hybrid_v4_eval_results,
    "official_score": official_score_v4,
    "paths": {
        "timeline_v3_progress_path": TIMELINE_V3_PROGRESS_PATH,
        "quality_v3_progress_path": QUALITY_V3_PROGRESS_PATH
    }
}

with open(HYBRID_V4_EVAL_PATH, "w", encoding="utf-8") as f:
    json.dump(save_obj, f, ensure_ascii=False, indent=2)

with open(HYBRID_V4_PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(hybrid_v4_predictions, f, ensure_ascii=False, indent=2)

print("\n✅ 已儲存 Hybrid v4 評估結果：", HYBRID_V4_EVAL_PATH)
print("✅ 已儲存 Hybrid v4 預測結果：", HYBRID_V4_PRED_PATH)

# ============================================================
# 17.7 Misleading 樣本檢查
# ============================================================

print("\n" + "=" * 90)
print("Misleading 樣本檢查")
print("=" * 90)

for item in test_data:
    if item.get("evidence_quality") == "Misleading":
        item_id = item["id"]
        pred = hybrid_v4_pred_by_id[item_id]

        print("id:", item_id)
        print("Gold verification_timeline:", item["verification_timeline"])
        print("Pred verification_timeline:", pred["verification_timeline"])
        print("Gold evidence_quality:", item["evidence_quality"])
        print("Pred evidence_quality:", pred["evidence_quality"])
        print("Gold promise_status:", item["promise_status"])
        print("Pred promise_status:", pred["promise_status"])
        print("Gold evidence_status:", item["evidence_status"])
        print("Pred evidence_status:", pred["evidence_status"])
        print("text:", get_text(item))